# Module 5 — Exercises (Student Worksheet)
### Hypergraph Networks & AllSet (the capstone) · TIFR ML School 2026

Student worksheet for the six exercises at the end of `Module5_Hypergraph_AllSet.ipynb`. **Self-contained**:
the *Setup* re-defines the `DeepSetsAgg` atom, the `AllSetLayer` (two nested DeepSets), the hypergraph tagger,
and loads JetClass as hypergraphs.

| # | Exercise | Idea it drives home |
|---|----------|---------------------|
| 1 | **Subjet hyperedges** | physically-motivated groups vs k-NN groups |
| 2 | **Hyperedge size** | k=1→graph, k=N→one DeepSet; where's the sweet spot |
| 3 | **Recover a GNN** | pair-hyperedges ⇒ AllSet *is* Module-2 message passing |
| 4 | **Sets of sets** | an event = jets = particles; grouping beats flattening |
| 5 | **Assignment** | scoring *groups* (V→E head) vs a SPANet-style attention baseline |
| 6 | **Attention pooling in AllSet** | swap the pool → the AllSet-*Transformer* |

> **A reuse trick the module gifts us:** `HypergraphNet` already has a `mode` switch — `"hyper"` (k-NN groups)
> vs `"deepsets"` (one group per graph). For the *sets-of-sets* exercise we feed it **event** hypergraphs and the
> same two modes become **jets-as-groups vs flatten-everything** — no new model needed.
>
> **Speed knobs:** `N_PER_CLS`, `EPOCHS`. Defaults run in a few minutes; raise for the module's headline
> hypergraph AUC (≈0.99).

> **How to use this worksheet.** Run the **Setup** section as-is (it loads the data and provides the models/helpers from the module). Then complete each exercise where you see `# TODO` and `raise NotImplementedError`. Read the **prompt** above each exercise — it sketches the approach. Check your work against the matching `*_Exercises_Solutions.ipynb`.

## Setup — the DeepSets atom, AllSet layer, hypergraph tagger, data

In [ ]:
import os, math, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torch_geometric.utils import scatter, softmax as geo_softmax
from torch_geometric.nn import global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score, roc_curve

torch.manual_seed(0); np.random.seed(0)
device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("device", device)

def mlp(i, o, h=64):
    return nn.Sequential(nn.Linear(i, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, o))

def knn_graph_native(x, k, batch):
    with torch.no_grad():
        N = x.size(0); x2 = (x*x).sum(1, keepdim=True)
        d2 = (x2 + x2.t() - 2*x @ x.t()).masked_fill(~(batch[None] == batch[:, None]), float("inf"))
        d2.fill_diagonal_(float("inf")); kk = min(k, max(N-1, 1))
        knn_d, idx = d2.topk(kk, dim=1, largest=False); valid = torch.isfinite(knn_d)
        centre = torch.arange(N, device=x.device).unsqueeze(1).expand(-1, kk)
        return torch.stack([idx[valid], centre[valid]], dim=0)

In [ ]:
# --- the Module-1 atom, and the AllSet layer = two nested DeepSets -----------------------
class DeepSetsAgg(nn.Module):
    """rho( POOL_over_group phi(x) ). pool in {'mean','sum','max','attn'} ('attn' used in Exercise 6)."""
    def __init__(self, in_dim, out_dim, reduce="mean"):
        super().__init__()
        self.phi = mlp(in_dim, out_dim); self.rho = mlp(out_dim, out_dim); self.reduce = reduce
        if reduce == "attn": self.score = nn.Linear(out_dim, 1)
    def forward(self, x, src, index, dim_size):
        v = self.phi(x)[src]
        if self.reduce == "attn":
            a = geo_softmax(self.score(v), index, num_nodes=dim_size)   # group-wise softmax over members
            agg = scatter(a * v, index, dim=0, dim_size=dim_size, reduce="sum")
        else:
            agg = scatter(v, index, dim=0, dim_size=dim_size, reduce=self.reduce)
        return self.rho(agg)

class AllSetLayer(nn.Module):
    """One hypergraph layer = V->E DeepSet then E->V DeepSet."""
    def __init__(self, dim, reduce="mean"):
        super().__init__()
        self.v2e = DeepSetsAgg(dim, dim, reduce); self.e2v = DeepSetsAgg(dim, dim, reduce)
        self.norm = nn.LayerNorm(dim)
    def forward(self, h, node_idx, edge_idx, num_nodes, num_edges):
        z = self.v2e(h, node_idx, edge_idx, num_edges)
        h = self.norm(h + self.e2v(z, edge_idx, node_idx, num_nodes))
        return h, z

class HypergraphNet(nn.Module):
    def __init__(self, in_feat, n_classes=2, dim=64, n_layers=3, reduce="mean"):
        super().__init__()
        self.embed = mlp(in_feat, dim)
        self.layers = nn.ModuleList([AllSetLayer(dim, reduce) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, n_classes))
    def forward(self, data, mode="hyper"):
        h = self.embed(data.x); batch = data.batch; N = h.size(0)
        if mode == "hyper":
            node_idx, edge_idx = data.hyperedge_index; num_edges = int(edge_idx.max()) + 1
        else:  # one hyperedge per graph (all its nodes) -> V->E pool == Deep Sets / PFN
            node_idx, edge_idx, num_edges = torch.arange(N, device=h.device), batch, int(batch.max()) + 1
        for layer in self.layers:
            h, _ = layer(h, node_idx, edge_idx, N, num_edges)
        return self.head(global_mean_pool(h, batch))

def n_params(m): return sum(p.numel() for p in m.parameters())

In [ ]:
# --- data: featurize jets (particles + POS), ready to assemble into hypergraphs ----------
import uproot, awkward as ak
CANDIDATE_PATHS = [
    "../data/JetClass_example_100k.root",
    "/Users/sanmay/Documents/ML_SCHOOLS/MLSCHOOL_2023_ICTS/Main_School/JetDataset/JetClass_example_100k.root",
    "./JetClass_example_100k.root",
]
root_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if root_path is None: raise FileNotFoundError("JetClass_example_100k.root not found")
print("Using:", root_path)

NPART, N_PER_CLS, K_HYPER, EPOCHS = 64, 3000, 8, 12
tree = uproot.open(root_path)["tree"]
br = tree.arrays(["part_px","part_py","part_pz","part_energy","part_deta","part_dphi",
                  "label_QCD","label_Tbqq","label_Tbl"])
is_qcd = ak.to_numpy(br["label_QCD"]).astype(bool)
is_top = ak.to_numpy(br["label_Tbqq"]).astype(bool) | ak.to_numpy(br["label_Tbl"]).astype(bool)
sel = np.concatenate([np.where(is_qcd)[0][:N_PER_CLS], np.where(is_top)[0][:N_PER_CLS]])
labels = np.concatenate([np.zeros(N_PER_CLS), np.ones(N_PER_CLS)]).astype(np.int64)
px,py,pz,e = br["part_px"][sel], br["part_py"][sel], br["part_pz"][sel], br["part_energy"][sel]
deta,dphi  = br["part_deta"][sel], br["part_dphi"][sel]
pt = np.sqrt(px**2+py**2); dR = np.sqrt(deta**2+dphi**2)
sumpt, sume = ak.sum(pt, axis=1), ak.sum(e, axis=1)
order = ak.argsort(pt, axis=1, ascending=False)                 # particles pT-sorted within each jet
def topN(a): return ak.to_numpy(ak.fill_none(ak.pad_none(a[order], NPART, clip=True), 0.0))
feat = [deta, dphi, np.log(pt+1e-8), np.log(e+1e-8), np.log(pt/sumpt+1e-8), np.log(e/sume+1e-8), dR]
X   = np.stack([topN(f) for f in feat], axis=-1).astype(np.float32)
POS = np.stack([topN(deta), topN(dphi)], axis=-1).astype(np.float32)
counts = np.minimum(ak.to_numpy(ak.num(pt, axis=1)), NPART)
realmask = (np.arange(NPART)[None,:] < counts[:,None]); mean, std = X[realmask].mean(0), X[realmask].std(0)+1e-6
X = (X - mean) / std
IN_FEAT = X.shape[-1]
print("featurized", len(labels), "jets, up to", NPART, "particles")

In [ ]:
# --- HyperData + builders (k-NN hyperedges) + train/eval --------------------------------
class HyperData(Data):
    def __inc__(self, key, value, *args, **kwargs):
        if key == "hyperedge_index":
            return torch.tensor([[self.num_nodes], [self.num_hyperedges]])
        return super().__inc__(key, value, *args, **kwargs)

def build_knn_hyper(x, pos, label, k):
    n = x.shape[0]
    ei = knn_graph_native(torch.from_numpy(pos), k, torch.zeros(n, dtype=torch.long))
    node_idx = torch.cat([torch.arange(n), ei[0]]); edge_idx = torch.cat([torch.arange(n), ei[1]])
    d = HyperData(x=torch.from_numpy(x), hyperedge_index=torch.stack([node_idx, edge_idx]),
                  y=torch.tensor([label], dtype=torch.long)); d.num_hyperedges = n
    return d

def make_loaders(dlist, bs=64):
    n = len(dlist); a, b = int(0.70*n), int(0.85*n)
    return {"train": DataLoader(dlist[:a], batch_size=bs, shuffle=True),
            "val":   DataLoader(dlist[a:b], batch_size=128),
            "test":  DataLoader(dlist[b:], batch_size=128)}

@torch.no_grad()
def evaluate(model, loader, mode="hyper"):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(device); ys.append(d.y.cpu()); ps.append(F.softmax(model(d, mode), -1)[:, 1].cpu())
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return {"auc": roc_auc_score(y, p), "y": y, "p": p}

def background_rejection(y, p, eff=0.5):
    fpr, tpr, _ = roc_curve(y, p)
    return 1.0 / max(np.interp(eff, tpr, fpr), 1.0/max(int((y==0).sum()), 1))

def train(model, loaders, mode="hyper", epochs=EPOCHS, lr=1e-3, quiet=True):
    model = model.to(device); opt = torch.optim.AdamW(model.parameters(), lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        model.train()
        for d in loaders["train"]:
            d = d.to(device); loss = F.cross_entropy(model(d, mode), d.y)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        if not quiet: print(f"  epoch {ep+1:2d}: val AUC {evaluate(model, loaders['val'], mode)['auc']:.4f}")
    return model

# reference k-NN hypergraph tagger, reused by exercises 1, 2, 6
knn_list = [build_knn_hyper(X[i, :int(counts[i])], POS[i, :int(counts[i])], int(labels[i]), K_HYPER)
            for i in range(len(labels)) if counts[i] >= 2]
import random; random.seed(0); random.shuffle(knn_list)
knn_loaders = make_loaders(knn_list)
torch.manual_seed(0); ref_hyper = train(HypergraphNet(IN_FEAT, dim=64, n_layers=3).to(device), knn_loaders)
res_ref = evaluate(ref_hyper, knn_loaders["test"])
print(f"reference k-NN hypergraph tagger: test AUC {res_ref['auc']:.4f}")
print("Setup complete.")

## Exercise 1 — Subjet hyperedges

> *Replace k-NN hyperedges with re-clustered subjets (assign each particle to its nearest of the 3 leading-pT
> particles). Do physically-motivated groups beat k-NN groups for top tagging?*

**The concept.** A k-NN hyperedge is generic — "a particle and its neighbours". A **subjet** hyperedge is
*physics*: a boosted top has ~3 prongs, so assigning every particle to its nearest of the **3 hardest** particles
carves the jet into its natural sub-structure (≈ its subjets). Since our particles are already pT-sorted, the "3
leading" are indices 0,1,2, and each particle joins the group of its nearest leading particle in (η, φ) — giving
**3 hyperedges** that *mean something*. We train the same AllSet net on these groups and compare.

In [ ]:
def build_subjet_hyper(x, pos, label, n_lead=3):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 2 — Hyperedge size

> *Scan K_HYPER (2, 4, 8, 16). At k=1 you approach a graph; at k=N a single DeepSet. Where is the sweet spot,
> and why?*

**The concept.** `K_HYPER` interpolates the whole course. Tiny `k` → each hyperedge is almost a pair → you are
back to a **graph** (Module 2). Huge `k` (→N) → each hyperedge is the whole jet → a single **DeepSet** (Module
1). In between, hyperedges are *local groups* — big enough to capture a prong/subjet, small enough to stay
specific. We rebuild the hypergraphs at several `k` and read off test AUC to locate the sweet spot.

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Exercise 3 — Recover a GNN

> *Make every hyperedge a pair {i,j} and confirm AllSet reproduces Module-2 message passing. (You've now shown
> DeepSets, GNNs and hypergraphs are one family.)*

**The concept.** Turn each k-NN **edge** `(i,j)` into a 2-node **hyperedge** `{i,j}`. Then AllSet's V→E step
aggregates exactly two nodes — a function of `(h_i, h_j)`, i.e. an **edge message** — and the E→V step
aggregates a node's incident edges — a **node update**. That is precisely the MPNN message→aggregate→update of
Module 2. So with pair-hyperedges, AllSet *is* a GNN; we confirm it trains to graph-like AUC.

In [ ]:
def build_pair_hyper(x, pos, label, k):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 4 — Sets of sets: an event hypergraph

> *Build an event hypergraph: nodes = particles, hyperedges = jets, plus one event-level hyperedge. Classify
> events and compare to flattening everything into one jet.*

**The concept.** Real analyses live at the **event** level: an event is a *set of jets*, each a *set of
particles* — a genuine "set of sets". A hypergraph expresses this natively: **jet** hyperedges group particles
into their jets, and one **event** hyperedge spans everything. We synthesize events from JetClass jets — a
**signal** event contains a top jet (+ QCD jets), a **background** event is all QCD — and classify them. The
comparison: keep the jet grouping (`mode="hyper"`) vs **flatten** all particles into one bag (`mode="deepsets"`,
which throws the jet structure away). Same network, so the delta isolates the value of the hierarchy.

In [ ]:
def build_event(jet_ids, label, J):
    """Concatenate J jets' particles; jet hyperedges (one per jet) + one event-level hyperedge spanning all."""
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 5 — Assignment: which particles came from which parent?

> *Set up a toy tt̄ jet→parent assignment and score candidate groupings with the V→E head; compare to a
> SPANet-style attention baseline.*

**The concept.** The deepest use of higher-order structure is **combinatorial assignment**: in `tt̄→6 jets`,
which 3 jets came from each top? A candidate parent is a **hyperedge** over its decay products, and the natural
score of a group is a **V→E DeepSet** of its members (permutation-invariant in the group, as it must be). We
build a toy: two tops (mass ≈ 173) each decaying to 3 massless jets, boosted/smeared, and the 6 jets shuffled.
The task: recover the correct 3+3 partition. We train the V→E head to score a triplet as "is this a real top?"
and, at test time, pick the partition whose two triplets score highest — then compare to a **SPANet-style**
self-attention scorer.

In [ ]:
from itertools import combinations
def make_ttbar(n_events, seed=0):
    """Two tops -> 3 massless jets each; return 6 shuffled jet 4-vectors + the truth partition (which top)."""
    rng = np.random.default_rng(seed); Ev, Truth = [], []
    for _ in range(n_events):
        jets, who = [], []
        for t in range(2):
            # 3 massless momenta summing to ~top mass in the top rest frame, then boost
            dirs = rng.normal(size=(3, 3)); dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)
            Emag = rng.uniform(40, 70, size=3); p = dirs * Emag[:, None]
            p -= p.mean(0)                                   # roughly balance so invariant mass ~ sum|E|
            E = np.linalg.norm(p, axis=1, keepdims=True); four = np.concatenate([E, p], 1)
            beta = rng.uniform(-0.6, 0.6, size=3); beta[2] += 0.3*t
            g = 1/np.sqrt(1-(beta**2).sum()); L = np.eye(4)
            L[0,0]=g; L[0,1:]=g*beta; L[1:,0]=g*beta; L[1:,1:]=np.eye(3)+(g-1)*np.outer(beta,beta)/(beta**2).sum()
            four = four @ L.T
            four += rng.normal(0, 2.0, four.shape)           # detector smearing
            jets += list(four); who += [t, t, t]
        idx = rng.permutation(6); jets = np.array(jets)[idx]; who = np.array(who)[idx]
        Ev.append(jets.astype(np.float32)); Truth.append(who)
    return Ev, Truth

def mink(a, b): return a[...,0]*b[...,0] - (a[...,1:]*b[...,1:]).sum(-1)
Ev_tr, Tr_tr = make_ttbar(3000, 0); Ev_te, Tr_te = make_ttbar(800, 1)
PARTS = [(set(c), set(range(6))-set(c)) for c in combinations(range(6), 3) if 0 in c]   # 10 unique 3+3 splits

def truth_partition(who):
    g0 = set(np.where(who == who[0])[0].tolist()); return (g0, set(range(6))-g0)

In [ ]:
class TripletScorer(nn.Module):
    def __init__(self, dim=48, spanet=False):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def encode(self, jets):                                   # jets (B,6,4) -> (B,6,dim)
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def score_group(self, h, group):                          # DeepSet over the 3 members of `group`
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
def batchify(Ev):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
def train_scorer(spanet, epochs=8):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
@torch.no_grad()
def assignment_accuracy(model):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 6 — Attention pooling inside AllSet (the AllSet-Transformer)

> *Swap the sum-pool in `DeepSetsAgg` for the attention pooling from Module 3 (this is literally the
> AllSet-Transformer variant).*

**The concept.** Each half of AllSet is a Deep Set, and (Module 1 Ex 5 / Module 3) a Deep Set's pool can be made
**learnable**: instead of a plain mean over a group's members, weight them by attention
`α = softmax(w·φ(x))` within the group. Doing this in *both* the V→E and E→V pools turns AllSet into the
**AllSet-Transformer** (Chien et al.'s stronger variant). Our `DeepSetsAgg(reduce="attn")` already implements it
with a group-wise `softmax` (PyG's segment `softmax`); we just build the net with `reduce="attn"` and compare.

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Wrap-up

You've reached the end of the worksheet. If every exercise runs without a `NotImplementedError`, you've implemented them all — compare your results and reasoning with the solutions notebook. The through-line across the course: **every model is constrained message passing** — a choice of relations, a permutation-invariant aggregation, and a baked-in symmetry.

In [ ]:
print("Module 5 exercise solutions complete.")